In [ ]:
# !pip install pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("classification_rf") \
    .getOrCreate()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
input_folder_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_pulito_finale.csv"

df = ss.read.csv(
    input_folder_path,
    header=True,
    inferSchema=True,
    sep=","
)

In [ ]:
df.show()

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile"]

df = df.drop(*columns_to_drop)

In [ ]:
df.show()

## Split data

In [ ]:
train_ratio = 0.8
test_ratio = 0.2

In [ ]:
# to stratify
fractions = {
    'Marine Debris': train_ratio,
    'Dense Sargassum': train_ratio,
    'Sparse Sargassum': train_ratio,
    'Natural Organic Material': train_ratio,
    'Ship': train_ratio,
    'Clouds': train_ratio,
    'Marine Water': train_ratio,
    'Sediment-Laden Water': train_ratio,
    'Foam': train_ratio,
    'Turbid Water': train_ratio,
    'Shallow Water': train_ratio,
    'Waves': train_ratio,
    'Cloud Shadows': train_ratio,
    'Wakes': train_ratio,
    'Mixed Water': train_ratio
}

seed = 42

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

df_with_id = df.withColumn("id", monotonically_increasing_id())

df_train = df_with_id.sampleBy("Class", fractions=fractions, seed=seed)
df_test  = df_with_id.join(df_train.select("id"), on="id", how="left_anti")

df_train = df_train.drop("id")
df_test  = df_test.drop("id")

## Confidence Conversion

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Confidence indexer as a stage of the Pipeline
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# OHE
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False
)

## Random search Cross Validation



In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]

label = "Class"

indexer = StringIndexer(inputCol=label, outputCol="label")

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# rf classificator
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

In [ ]:
import random
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create a pipeline just to prepare the data
prep_pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler])

prep_model = prep_pipeline.fit(df_train)
df_train_prep = prep_model.transform(df_train)

In [ ]:
df_train_prep.cache()
print(f"Training records: {df_train_prep.count()}")

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

large_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [4, 6, 8])
    .addGrid(rf.subsamplingRate, [0.7, 0.8, 1.0])
    .addGrid(rf.numTrees, [50, 100])
    .build()
)

NUM_SAMPLES = 8
SEED = 42
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)

crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)

cvModel = crossval.fit(df_train_prep)

# Results
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics

print(f"Best F1-score: {max(avgMetrics):.4f}")

print("Detail 8 combinations tested:")
for params, metric in zip(sampled_paramGrid, avgMetrics):
    print(f"F1 Score: {metric:.4f}")
    for param, value in params.items():
        print(f"   - {param.name}: {value}")
    print("-" * 30)

In [ ]:
# saving path
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_rf_model"

# best model
best_pipeline_model = cvModel.bestModel

# Save model (on Drive)
print(f"Saving model in: {OUTPUT_PATH} ...")
best_pipeline_model.write().overwrite().save(OUTPUT_PATH)

In [ ]:
from pyspark.sql.functions import col

df_test_prep = prep_model.transform(df_test)

# Apply the best model to the test set (df_test)
predictions = bestModel.transform(df_test_prep)
predictions.cache()
predictions.count()

# key columns for the evaluation
predictions.select(col("label"), col("prediction")).show(5)

metrics_to_evaluate = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]

results = {}
for metric_name in metrics_to_evaluate:
    # Update the evaluator for the current metric
    test_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )

    # Compute metric
    metric_value = test_evaluator.evaluate(predictions)

    results[metric_name] = metric_value
    print(f" {metric_name.ljust(20)}: {metric_value:.4f}")

confusion_matrix = predictions.groupBy('label').pivot('prediction').count().fillna(0).orderBy('label')

confusion_matrix.show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


class_labels = prep_model.stages[2].labels

# Restore original labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=class_labels
)

predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()

support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()

# Prepare RDD for MulticlassMetrics
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)

metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()

print(f"{'Class':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)

for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall = metrics.recall(label=label_index)
    class_f1 = metrics.fMeasure(label=label_index)

    class_support = support_map.get(label_index, 0)

    class_string = class_labels[int(label_index)]

    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")

predictions_with_strings.unpersist()
predictions.unpersist()

In [ ]:
indexer_model = prep_model.stages[2]

for index, label in enumerate(indexer_model.labels):
    print(f"Indice {index}: {label}")